
## 1. What is Data Ingestion?


**Data ingestion** is the process of` collecting data from different sources` and `loaded it into a data platform` for further processing.


**In my E-Commerce Lakehouse project:** 

- I work with seven datasets, including customers, products, orders, payments and shipments. 
- I ingest the incoming CSV files into Databricks using Auto Loader.
- Store the raw data in Bronze tables.



## 2. What is Medallion Architecture?


**Medallion Architecture** is a `data architecture` used to organizes data into three layers: `Bronze, Silver and Gold`.


**In my project:**

- Bronze stores raw data, 
- Silver handles cleaning, 
- validation and deduplication, 
- Gold contains business-ready data such as daily sales and product performance. 
- This separation makes the pipeline easier to maintain and troubleshoot.


## 2.1 What is the Bronze Layer?


- **Bronze layer** is the `raw data layer`. 
- I ingest `source files` into `Bronze Delta tables` with `minimal transformation`.
- I preserve the original source records so that I can audit the data or reprocess it if any issue occurs in downstream layers.


## 2.2 What is the Silver Layer?


**Silver layer** is the `cleaned` and `validated data layer`.

- I perform operations such as 
    - handling null values, 
    - removing duplicates, 
    - converting data types, 
    - standardizing columns,
    - validating business rules. 

I also separate `invalid records` into `quarantine` when required.



## 2.3 What is the Gold Layer?


**Gold layer** is the `business-ready layer` used for `reporting`,`dashboard` and `analytics`.

**For example:**
 - In my project, 
    - I create datasets such as 
                            - daily sales, 
                            - product performance, 
                            - customer lifetime value 
                            - inventory status. 
   -  These tables are consumed by Databricks SQL dashboards.


## 3. What is the difference between Bronze, Silver and Gold?


- Bronze stores `raw source data`, 
- Silver stores `cleaned` and `validated data`, 
- Gold stores `business-ready data`.

**For example:**
   - when I ingest customer data, 
       - I first preserve the original records in Bronze, 
       - clean and deduplicate them in Silver, 
       - use the trusted data for business reporting in Gold.



## 4. What is Auto Loader?


**Auto Loader** is a `Databricks feature` used to automatically load `new files from cloud storage` into `Delta Lake` using **Spark Structured Streaming**.

- In my project, 
    - I use the `cloudFiles` format to discover incoming CSV files. 
    - I configure a schema location and checkpoint location to manage    schema information and processing progress. 
    - I use `availableNow` when I want to process available files and stop the stream.


## 4.1 What is Directory Listing Mode?


- **Directory Listing Mode** is used to `discovers new files by listing the source directory` and checking which `files have not been processed`.
- It is useful when a `simpler file-discovery approach` is sufficient.



## 4.2 What is File Notification Mode?


- **File Notification Mode** is used to `discovers new files through cloud storage notifications` instead of `relying entirely on directory listing`.
- It is useful for `large-scale ingestion` because notifications can reduce the overhead of repeatedly listing directories.


## 5. What is COPY INTO?


- **COPY INTO** is a `Databricks SQL command` used to `incrementally load files from cloud storage` into an `existing Delta table`.
- It maintains file load history and normally avoids loading the same files again. I would use it for a straightforward SQL-based batch ingestion process.


## 5.1 What is the difference between Auto Loader and COPY INTO?


- **Auto Loader** is a `Databricks feature` used to automatically load `new files from cloud storage` into `Delta Lake` using **Spark Structured Streaming**.
- **COPY INTO** is a `Databricks SQL command` used to `incrementally load files from cloud storage` into an `existing Delta table`.


## 6. What is Incremental Loading?


- Incremental loading means processing only new or changed data instead of loading the complete dataset every time.
- For example, if my orders table receives 8,000 new records today, I process those new records rather than reprocessing the entire historical dataset.
- I can use Auto Loader to discover new files and Delta MERGE to apply row-level inserts and updates when needed.


## 7. What is Idempotency?


Idempotency means that running the same pipeline again should not create duplicate records or incorrect results.

For example, if my orders pipeline fails and the same batch is retried, I should not insert the same orders twice. I handle this using checkpoints, business-key deduplication and Delta MERGE.


## 7.1 How do you implement Idempotency?


I use a combination of checkpoint management, deduplication using business keys and Delta MERGE.

Before merging, I ensure that the source contains only one record per business key. I also make sure that retries do not repeatedly apply incorrect updates.


## 8. What is the difference between a Traditional Data Warehouse and Medallion Architecture?


- A traditional data warehouse often follows ETL, where data is transformed before loading into the target warehouse.
- In a Databricks lakehouse, we commonly follow ELT. We first ingest raw data into Bronze, clean it in Silver and create business-ready datasets in Gold.
- However, both architectures can use ETL or ELT depending on the requirements.


## 8.1 What is ETL?


ETL stands for Extract, Transform and Load.

We extract data from source systems, transform it by cleaning and processing it, and then load it into the target system.


## 8.2 What is ELT?


ELT stands for Extract, Load and Transform.

We first load raw data into the lakehouse and then perform transformations using tools such as PySpark and Spark SQL.


## 9. How do you map a Traditional Data Warehouse to Medallion Architecture?


Conceptually, a traditional staging area maps to Bronze, an operational or integrated data layer maps mainly to Silver, and data marts map to Gold.

This is not a strict one-to-one mapping. The actual design depends on the business requirements and existing data architecture.





## 10. How would you design an end-to-end Databricks pipeline using Auto Loader and Medallion Architecture?


- First, I identify the source systems and datasets. In my E-Commerce Lakehouse project, I work with seven datasets.
- I use Auto Loader to incrementally ingest incoming files into Bronze Delta tables. Then I process the Bronze data in Silver by handling duplicates, null values, data types and business validations.
- For incremental updates, I use Delta MERGE with appropriate business keys. Finally, I create Gold tables for daily sales, product performance, customer lifetime value and other business metrics.
- I orchestrate the pipeline using Databricks Workflows and use checkpoints, logging and monitoring to handle failures and retries.


## 10.1 Why do you use Auto Loader in your project?


I use Auto Loader because it automatically discovers and incrementally processes new files. It avoids manually identifying files for every ingestion run and provides checkpoint-based progress tracking.


## 10.2 Why do you use Delta Tables?


I use Delta tables because they provide ACID transactions, schema enforcement, schema evolution and features such as MERGE and Time Travel. These capabilities help maintain reliable data pipelines.


## 11. What is the difference between %run and dbutils.notebook.run()?

`%run` executes another notebook and makes its variables and functions available in the current notebook context. I use it to reuse configurations and common functions.

`dbutils.notebook.run()` executes another notebook as a separate notebook run. It supports passing parameters and returning a result using `dbutils.notebook.exit()`.

For production orchestration, I prefer Databricks Workflows because they support task dependencies, retries and monitoring.



## 12. How would you refactor a monolithic notebook?

- First, I identify the different responsibilities in the notebook and separate them into reusable components.
- I move configurations into a separate notebook, common logic into reusable functions, and create separate notebooks for Bronze, Silver and Gold processing.
- Then I orchestrate these notebooks using Databricks Workflows. This improves maintainability, testing, debugging and reusability.


## 13. How would you build a generic framework to ingest 100 datasets?


- I would build a configuration-driven ingestion framework instead of creating 100 separate notebooks.
- I maintain dataset-specific configurations such as source path, file format, primary key, load type and target table. 
- Then I create reusable functions that read the configuration and process each dataset dynamically.
- This reduces duplicated code and makes it easier to add new datasets without changing the core framework.